This was an auxilliary notebook that was used to figure out how to tune hyperparameter factors so that all models would have similar
numbers of trainable parameters. It is not part of the main codebase, but it is included here for completeness. It is not meant to be run, but it can be run if you want to see how the hyperparameters were tuned.

In [1]:
import torch
import torch.nn as nn
import sys
sys.path.append('/home/stein/classes/REAN')
from rean.models.CNN import PlainCNN
from rean.models.P4 import P4CNN
from rean.models.RelaxedP4 import RelaxedP4CNN

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [2]:
#set base tunable hyperparameters, which will be applied unmodified to the plain CNN. they
# will be adapted for the RelaxedP4CNN  of for P4 to keep parameter count similar.
in_channels = 1
hidden_dim = 20
out_channels = hidden_dim #no motivation to do this any differently for now
classes = 10
kernel_size = 3
group_order = 4
num_gconvs = 6
print(hidden_dim//3.2)

6.0


In [3]:
CNN = PlainCNN(in_channels = in_channels,
                  hidden_dim = hidden_dim,
                  out_channels = out_channels,
                  classes = classes,
                    num_gconvs = num_gconvs,
                  kernel_size = kernel_size)
P4CNN_model = P4CNN(in_channels = in_channels,
                    hidden_dim = hidden_dim,
                    out_channels = out_channels,
                    classes = classes,
                    num_gconvs = num_gconvs,
                    kernel_size = kernel_size,
                    group_order = group_order)

RelaxedP4CNN_model = RelaxedP4CNN(in_channels = in_channels,
                                  hidden_dim = int(hidden_dim,),
                                  out_channels = int(out_channels),
                                  num_gconvs= num_gconvs,
                                  classes = classes,
                                  kernel_size = kernel_size,
                                  group_order = group_order)

25400 trainable parameters in P4CNN model
27544 trainable parameters in RelaxedP4CNN model


In [4]:
models = {
    "Plain CNN": CNN,
    "P4 CNN": P4CNN_model,
    "Relaxed P4 CNN": RelaxedP4CNN_model}

for name, model in models.items():
    param_count = count_parameters(model)
    print(f"{name} has {param_count} trainable parameters.")

Plain CNN has 25750 trainable parameters.
P4 CNN has 25400 trainable parameters.
Relaxed P4 CNN has 27544 trainable parameters.


So, best practice seems to be dividing the hidden and output dimensions by 2 = rt(4) for P4CNNs, and by about 3.2 (higher for higher hidden dim numbers) for RelaxedP4CNNs, to keep parameter counts similar.